## 📦 Setup

In [ ]:
import os
import sys
import torch
import librosa
import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
from transformers import AutoFeatureExtractor, HubertModel
import torch.nn as nn

from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)

os.chdir('/content/multimodal-emotion-recognition')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Load model
class HuBERTForEmotion(nn.Module):
    def __init__(self, num_labels=7):
        super(HuBERTForEmotion, self).__init__()
        self.hubert = HubertModel.from_pretrained('facebook/hubert-base-ls960')
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.hubert.config.hidden_size, num_labels)

    def forward(self, input_values):
        outputs = self.hubert(input_values)
        pooled = outputs.last_hidden_state.mean(dim=1)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

feature_extractor = AutoFeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
model = HuBERTForEmotion().to(device)
model.load_state_dict(torch.load('models/checkpoints/hubert_best.pth'))
model.eval()

emotions = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust', 'surprise']
print("✅ Model loaded!")

## 🎵 Test 1: Upload Audio File

In [ ]:
print("🎤 Upload an audio file\n")

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"Processing: {filename}")

    # Load audio
    y, sr = librosa.load(filename, sr=16000)

    # Preprocess
    inputs = feature_extractor(y, sampling_rate=sr, return_tensors='pt').to(device)

    # Predict
    with torch.no_grad():
        outputs = model(inputs['input_values'])
        probabilities = torch.softmax(outputs, dim=-1)[0]
        predicted_idx = outputs.argmax(-1).item()
        confidence = probabilities[predicted_idx].item()

    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    # Waveform
    ax1.plot(y[:sr*3])  # First 3 seconds
    ax1.set_title(f"🎙️ Audio: {filename}\n🎭 Detected: {emotions[predicted_idx].upper()} ({confidence*100:.1f}%)",
                  fontsize=12, fontweight='bold')
    ax1.set_xlabel('Samples')
    ax1.set_ylabel('Amplitude')

    # Predictions
    colors = ['red' if i == predicted_idx else 'skyblue' for i in range(7)]
    ax2.barh(emotions, probabilities.cpu().numpy(), color=colors)
    ax2.set_xlabel('Probability')
    ax2.set_xlim(0, 1)

    plt.tight_layout()
    plt.show()

    print(f"\n🎭 Prediction: {emotions[predicted_idx]} ({confidence*100:.1f}%)\n")

## 🌐 Test 2: Gradio Web Interface

In [ ]:
def predict_speech_emotion(audio_file):
    """Predict emotion from audio file"""
    if audio_file is None:
        return {"Error": "Please upload audio"}

    # Load audio
    y, sr = librosa.load(audio_file, sr=16000)

    # Feature extraction
    inputs = feature_extractor(y, sampling_rate=sr, return_tensors='pt').to(device)

    # Predict
    with torch.no_grad():
        outputs = model(inputs['input_values'])
        probabilities = torch.softmax(outputs, dim=-1)[0]

    emotion_dict = {emotions[i]: float(probabilities[i].cpu().numpy()) for i in range(7)}
    return emotion_dict

# Gradio interface
with gr.Blocks(title="🎙️ Speech Emotion Recognition", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🎙️ Speech Emotion Recognition

    **Upload audio to detect emotions!**

    Trained with HuBERT on RAVDESS dataset
    - Accuracy: 80%+
    - 7 emotion classes
    """)

    with gr.Row():
        with gr.Column():
            audio_input = gr.Audio(type="filepath", label="🎤 Upload Audio")
        with gr.Column():
            emotion_output = gr.Label(num_top_classes=7, label="😊 Predictions")

    gr.Button("🔍 Analyze").click(
        fn=predict_speech_emotion,
        inputs=audio_input,
        outputs=emotion_output
    )

demo.launch(share=True)

## ✅ Phase 3 Complete! 🎉

✅ HuBERT trained for speech emotion
✅ Achieved 80%+ accuracy
✅ Interactive audio testing working

**Both Phase 2 & 3 are now complete!** 🚀